In [ ]:
!pip install torch==2.2.1 scikit-learn pandas numpy tqdm --quiet

In [3]:
import pandas as pd, numpy as np, torch
from sklearn.preprocessing import StandardScaler
from pathlib import Path

DATA_DIR   = Path("data")        # <- adjust
TRAIN_CSV  = DATA_DIR/"train_dataset.csv"
TEST_CSV   = DATA_DIR/"test_features.csv"  # same 35 024 rows, no target

train_df = pd.read_csv(TRAIN_CSV, parse_dates=["date"]).copy()
test_df  = pd.read_csv(TEST_CSV,  parse_dates=["date"]).copy()


RAW_COLS   = [
    "temperature_exterieure",
    "humidite",
    "ensoleillement",
    "vitesse_vent",
    "direction_vent",
    "temperature_interieure"
]

HORIZON   = 16        # hours_ahead*4 if you ever change sampling rate
LOOKBACK  = 16
STRIDE_TR = 1         # slide one step at a time during training

# ---------- helpers ----------------------------------------------------------
def add_time_features(df):
    t     = pd.to_datetime(df["date"])            # 15‑min resolution
    secs  = t.dt.hour*3600 + t.dt.minute*60
    dow   = t.dt.dayofweek
    df["hour_sin"] = np.sin(2*np.pi*secs/86400)
    df["hour_cos"] = np.cos(2*np.pi*secs/86400)
    df["dow_sin"]  = np.sin(2*np.pi*dow/7)
    df["dow_cos"]  = np.cos(2*np.pi*dow/7)
    df["is_peak"]  = ((t.dt.hour.between(6,8)) | (t.dt.hour.between(16,18))).astype(int)
    return df

train_df_full = add_time_features(pd.read_csv(TRAIN_CSV))
test_df  = add_time_features(pd.read_csv(TEST_CSV))

train_df = train_df_full[:-16]


LAGS     = [1, 4, 16]   # 15 min, 1 h, 4 h
ROLL_WIN = 16           # 4 h window

def add_lag_features(df, base_cols):
    df = df.copy()                            # safe guard
    for col in base_cols:
        for l in LAGS:
            # instead of df[f"..."] = … do:
            df.loc[:, f"{col}_lag{l}"] = df[col].shift(l)
        df.loc[:, f"{col}_roll{ROLL_WIN}"] = (
            df[col].shift(1)
                 .rolling(ROLL_WIN, min_periods=1)
                 .mean()
        )
    return df


train_df = add_lag_features(train_df, RAW_COLS)
test_df  = add_lag_features(test_df,  RAW_COLS)

# train_df.shape, train_df.tail()

# FEAT_COLS = RAW_COLS + ["hour_sin","hour_cos","dow_sin","dow_cos","is_peak"]
# rebuild FEAT_COLS:
FEAT_COLS = (
    RAW_COLS
    + [c for c in train_df.columns if c.startswith(("hour_", "dow_", "is_peak"))]
    + [f"{col}_lag{l}" for col in RAW_COLS for l in LAGS]
    + [f"{col}_roll{ROLL_WIN}" for col in RAW_COLS]
)

# ---------- scaling ----------------------------------------------------------
x_scaler = StandardScaler()
y_scaler = StandardScaler()

train_df[FEAT_COLS] = x_scaler.fit_transform(train_df[FEAT_COLS])
train_df["y_scaled"] = y_scaler.fit_transform(
    train_df[["puissance_cvac_future"]]
)

train_df = train_df.dropna(subset=FEAT_COLS).reset_index(drop=True)

# train_df["y_scaled"] = train_df["puissance_cvac_future"]

test_df[FEAT_COLS]  = x_scaler.transform(test_df[FEAT_COLS])

In [9]:
# class SeqDataset(torch.utils.data.Dataset):
#     def __init__(self, dat, feat_cols, target_col=None, L=16, H=16, stride=1):
#         self.X = dat[feat_cols].values.astype("float32")
#         self.has_targets = target_col is not None
#         self.L, self.H, self.stride = L, H, stride
#         self.seq_count = (len(dat) - L - H) // stride + 1
#         if self.has_targets:
#             self.y = dat[target_col].values.astype("float32")

#     def __len__(self): return self.seq_count

#     def __getitem__(self, idx):
#         i = idx * self.stride
#         x = self.X[i : i+self.L]
#         if self.has_targets:
#             y = self.y[i+1 : i+1+self.H]
# #             y = self.y[i+self.L : i+self.L+self.H] 
#             return torch.from_numpy(x), torch.from_numpy(y)
#         else:
#             return torch.from_numpy(x), torch.zeros(self.H)  # placeholder


# class SeqDataset(torch.utils.data.Dataset):
#     def __init__(self, dat, feat_cols, target_col, L, H, stride=1):
#         self.X = dat[feat_cols].values.astype("float32")
#         self.y = dat[target_col].values.astype("float32")
#         self.L, self.H, self.stride = L, H, stride
#         self.seq_count = (len(dat) - L - H) // stride + 1

#     def __len__(self): return self.seq_count

#     def __getitem__(self, idx):
#         i = idx * self.stride
#         x = self.X[i : i+self.L]        # shape (L, F)
        
#         y = self.y[i+self.L : i+self.L+self.H] 
# #         y = self.y[i+1 : i+1+self.H]    # predict t+1 … t+H
#         return torch.from_numpy(x), torch.from_numpy(y)

class SeqDataset(torch.utils.data.Dataset):
    def __init__(self, dat, feat_cols, target_col=None, L=16, H=16, stride=1):
        # features
        self.X = dat[feat_cols].values.astype("float32")
        # do we have targets?
        self.has_targets = target_col is not None
        if self.has_targets:
            self.y = dat[target_col].values.astype("float32")
        
        self.L, self.H, self.stride = L, H, stride
        self.seq_count = (len(dat) - L - H) // stride + 1

    def __len__(self):
        return self.seq_count

    def __getitem__(self, idx):
        i = idx * self.stride
        x = self.X[i : i + self.L]               # (L, F)
        if self.has_targets:
            y = self.y[i + 1 : i + 1 + self.H]   # (H,)
            return (
                torch.from_numpy(x),
                torch.from_numpy(y).unsqueeze(1),  # (H,)->(H,1) if your model expects that
            )
        else:
            # no target: return dummy tensor so unpack (x,_) still works
            return torch.from_numpy(x), torch.zeros(self.H, 1, dtype=torch.float32)


# temporal split: first 80 % train, last 20 % val
N_total = len(train_df)
split   = int(N_total*0.8)
train_set = SeqDataset(train_df.iloc[:split], FEAT_COLS, "y_scaled",
                       LOOKBACK, HORIZON, STRIDE_TR)
val_set   = SeqDataset(train_df.iloc[split:], FEAT_COLS, "y_scaled",
                       LOOKBACK, HORIZON, STRIDE_TR)

train_loader = torch.utils.data.DataLoader(train_set,  batch_size=128,
                                           shuffle=False, drop_last=True)
val_loader   = torch.utils.data.DataLoader(val_set,    batch_size=128,
                                           shuffle=False, drop_last=False)


In [5]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, feat_dim, hidden_dim, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(feat_dim, hidden_dim, n_layers,
                            batch_first=True, dropout=dropout)

    def forward(self, x):
        # x : (B, L, F)
        outputs, (h,c) = self.lstm(x)
        return h, c          # final states only

class Decoder(nn.Module):
    def __init__(self, hidden_dim, out_dim=1, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(out_dim, hidden_dim, n_layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden_dim, out_dim)

    def forward(self, y0, h, c, steps, teacher=None, tf_ratio=1.0):
        # y0 : (B, 1) initial input (previous target or 0)
        outs = []
        inp  = y0
        for t in range(steps):
            out, (h,c) = self.lstm(inp.unsqueeze(1), (h,c))
            pred = self.fc(out.squeeze(1))  # (B,1)
            outs.append(pred)
            if teacher is not None and torch.rand(1)<tf_ratio:
                inp = teacher[:,t:t+1]      # ground‑truth
            else:
                inp = pred.detach()
        return torch.cat(outs, dim=1)       # (B, H)

class Seq2Seq(nn.Module):
    def __init__(self, feat_dim, hidden_dim=128, n_layers=2, dropout=0.2):
        super().__init__()
        self.encoder = Encoder(feat_dim, hidden_dim, n_layers, dropout)
        self.decoder = Decoder(hidden_dim, out_dim=1, n_layers=n_layers, dropout=dropout)

    def forward(self, src, tgt=None, tf_ratio=1.0):
        h,c  = self.encoder(src)
        y0   = torch.zeros(src.size(0), 1, device=src.device)
        preds = self.decoder(y0, h, c, HORIZON, teacher=tgt, tf_ratio=tf_ratio)
        return preds


In [ ]:
# # ──────────────────────────────────────────────────────────────────────────────
# # 2) SEQ2SEQ MODEL WITH LUONG ATTENTION
# # ──────────────────────────────────────────────────────────────────────────────
# import torch.nn as nn
# import torch

# class Encoder(nn.Module):
#     def __init__(self, feat_dim, hidden_dim, n_layers=2, dropout=0.2):
#         super().__init__()
#         self.lstm = nn.LSTM(
#             feat_dim, hidden_dim, n_layers,
#             batch_first=True, dropout=dropout, bidirectional=False
#         )

#     def forward(self, x):
#         # x: (B, L, F)
#         outputs, (h, c) = self.lstm(x)
#         # outputs: (B, L, H)  — we need these for attention
#         return outputs, (h, c)

# class Decoder(nn.Module):
#     def __init__(self, hidden_dim, out_dim=1, n_layers=2, dropout=0.2):
#         super().__init__()
#         self.lstm = nn.LSTM(
#             out_dim, hidden_dim, n_layers,
#             batch_first=True, dropout=dropout
#         )
#         # since we'll concat [decoder_out ; context], fc sees 2*H
#         self.fc   = nn.Linear(hidden_dim * 2, out_dim)

#     def forward(self, y0, h, c, steps, encoder_outputs, tf_ratio=1.0):
#         # y0: (B,1)   initial input
#         # h,c: decoder init states
#         # encoder_outputs: (B, L, H)
#         outs = []
#         inp  = y0
#         B, L, H = encoder_outputs.size()
#         for t in range(steps):
#             out, (h, c) = self.lstm(inp.unsqueeze(1), (h, c))
#             out = out.squeeze(1)               # (B, H)
#             # Luong dot‑product attention
#             # use top layer hidden state:
#             h_last = h[-1]                     # (B, H)
#             # compute scores: (B, L)
#             scores = torch.bmm(encoder_outputs, h_last.unsqueeze(-1)).squeeze(-1)
#             α      = torch.softmax(scores, dim=1)                   # (B, L)
#             context= torch.bmm(α.unsqueeze(1), encoder_outputs).squeeze(1)  # (B, H)
#             # concat & predict
#             combined = torch.cat([out, context], dim=1)             # (B, 2H)
#             pred     = self.fc(combined)                            # (B, 1)
#             outs.append(pred)
#             # next input
#             if self.training and torch.rand(1, device=pred.device) < tf_ratio:
#                 inp = tgt[:, t:t+1]                                 # teacher forcing
#             else:
#                 inp = pred.detach()
#         return torch.cat(outs, dim=1)                              # (B, steps)

# class Seq2Seq(nn.Module):
#     def __init__(self, feat_dim, hidden_dim=128, n_layers=2, dropout=0.2):
#         super().__init__()
#         self.encoder = Encoder(feat_dim, hidden_dim, n_layers, dropout)
#         self.decoder = Decoder(hidden_dim, out_dim=1, n_layers=n_layers, dropout=dropout)

#     def forward(self, src, tgt=None, tf_ratio=1.0):
#         # src: (B, L, F)
#         encoder_outputs, (h, c) = self.encoder(src)
#         y0 = torch.zeros(src.size(0), 1, device=src.device)
#         # pass encoder outputs into decoder
#         return self.decoder(y0, h, c, HORIZON, encoder_outputs, tf_ratio=tf_ratio)


In [6]:
# from tqdm.auto import tqdm
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = Seq2Seq(len(FEAT_COLS)).to(device)
optim  = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, factor=0.5,
                                                    patience=3, verbose=True)
loss_fn = nn.MSELoss()

EPOCHS        = 100
TEACHER_DECAY = lambda e: max(0.1, 0.95 ** e)
# TEACHER_DECAY = lambda e: max(0.0, 1 - e/25)   # 1.0 → 0.0 by epoch 25
CLIP_NORM     = 1.0

def run_epoch(loader, train=True, epoch=0):
    model.train(train)
    total = 0
    pbar  = tqdm(loader, leave=False)
    for x,y in pbar:
        x,y = x.to(device), y.to(device)
        tf  = TEACHER_DECAY(epoch) if train else 0.0
        with torch.set_grad_enabled(train):
            pred = model(x, tgt=y, tf_ratio=tf)
            loss = loss_fn(pred, y)
            if train:
                optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                optim.step()
        total += loss.item() * len(x)
        pbar.set_description(f"{'Train' if train else 'Val'} loss {loss.item():.4f}")
    return total / len(loader.dataset)

best_val = float("inf")
patience = 8
for epoch in range(1, EPOCHS+1):
    tr = run_epoch(train_loader, train=True,  epoch=epoch)
    vl = run_epoch(val_loader,   train=False, epoch=epoch)
    sched.step(vl)
    print(f"Epoch {epoch:02d} | train {tr:.4f} | val {vl:.4f}")
    if vl < best_val:
        best_val, best_state = vl, model.state_dict().copy()
        patience = 8
    else:
        patience -= 1
        if patience == 0:
            print("Early stopping!")
            break

model.load_state_dict(best_state)
torch.save({"model":model.state_dict(),
            "x_scaler":x_scaler, "y_scaler":y_scaler},
           DATA_DIR/"seq2seq_hvac1.pt")

/home/jae/.local/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 01 | train 0.3054 | val 1.9308


Epoch 02 | train 0.1239 | val 0.9983


Epoch 03 | train 0.0361 | val 0.9023


Epoch 04 | train 0.0289 | val 0.7862


Epoch 05 | train 0.0283 | val 0.6193


... (62 outputs left out)

Epoch 68 | train 0.0342 | val 0.1626


Epoch 69 | train 0.0323 | val 0.1627


Epoch 70 | train 0.0312 | val 0.1627


Epoch 71 | train 0.0325 | val 0.1627
Early stopping!


In [13]:
def predict_test_set(model, test_df, feat_cols, lookback=16, horizon=16):
    # Create dataset for test data
    test_set = SeqDataset(test_df, feat_cols, target_col=None, 
                          L=lookback, H=horizon, stride=1)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=128, 
                                             shuffle=False, drop_last=False)
    
    model.eval()
    all_preds = []
    
    with torch.no_grad():
        for x, _ in tqdm(test_loader, desc="Generating predictions"):
            x = x.to(device)
            # Forward pass without teacher forcing
            preds = model(x, tgt=None, tf_ratio=0.0)
            # Scale predictions back to original range
            preds = y_scaler.inverse_transform(preds.cpu().numpy())
            all_preds.append(preds)
    
    # Concatenate all predictions
    all_preds = np.vstack(all_preds)
    
    # Create a DataFrame for the predictions
    # Note: We need to handle the lookback period with NaNs
    predictions = np.full(len(test_df), np.nan)
    
    # Fill in predictions (offset by lookback period)
    # For each starting point i, we've predicted horizon steps ahead
    for i in range(len(all_preds)):
        # The prediction starts at position i+lookback in the original test set
        for h in range(horizon):
            if i+lookback+h < len(predictions):
                # Take the first prediction for each timestamp (could also average overlapping predictions)
                if np.isnan(predictions[i+lookback+h]):
                    predictions[i+lookback+h] = all_preds[i, h]
    
    # Create submission dataframe
    submission = pd.DataFrame({
        'ID': test_df['ID'],
        'puissance_cvac_future': predictions
    })
    
    # Handle missing values at beginning (no prediction for first lookback points)
    # You might want to fill these with a simple model or copy from training data
    # Here we'll just forward-fill the first available prediction
    submission['puissance_cvac_future'] = submission['puissance_cvac_future'].fillna(method='bfill')
    
    return submission

# Load the trained model
checkpoint = torch.load(DATA_DIR/"seq2seq_hvac1.pt")
model.load_state_dict(checkpoint['model'])
model.to(device)

# Generate predictions
submission_df = predict_test_set(model, test_df, FEAT_COLS, LOOKBACK, HORIZON)

# Save predictions to CSV
submission_df.to_csv(DATA_DIR/"submission.csv", index=False)

Generating predictions: 100%|█████████████████| 274/274 [00:05<00:00, 50.80it/s]
/tmp/ipykernel_20135/516556968.py:46: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  submission['puissance_cvac_future'] = submission['puissance_cvac_future'].fillna(method='bfill')


In [12]:
submission_df[:10]

,date,puissance_cvac_future
0,2022-01-01 00:00:00,5.586432
1,2022-01-01 00:15:00,5.586432
2,2022-01-01 00:30:00,5.586432
3,2022-01-01 00:45:00,5.586432
4,2022-01-01 01:00:00,5.586432
5,2022-01-01 01:15:00,5.586432
6,2022-01-01 01:30:00,5.586432
7,2022-01-01 01:45:00,5.586432
8,2022-01-01 02:00:00,5.586432
9,2022-01-01 02:15:00,5.586432


In [ ]:
infer_set = SeqDataset(
    test_df,
    FEAT_COLS,
    target_col=None,  # no targets!
    L=LOOKBACK,
    H=HORIZON,
    stride=1           # one prediction per row
)

infer_loader = torch.utils.data.DataLoader(
    infer_set,
    batch_size=256,
    shuffle=False,
    drop_last=False
)

In [ ]:
early_preds = []

with torch.inference_mode():
    for i in range(LOOKBACK):
        pad_len = LOOKBACK - i
        pad     = np.zeros((pad_len, len(FEAT_COLS)), dtype=np.float32)
        seq     = test_df[FEAT_COLS].iloc[:i].values.astype(np.float32)
        x_padded = np.vstack([pad, seq])  # shape (16, F)
        x_tensor = torch.from_numpy(x_padded).unsqueeze(0).to(device)  # shape (1, L, F)

        y_hat = model(x_tensor, tgt=None, tf_ratio=0.0)  # shape (1, HORIZON)
        y_first = y_hat[:, 0]                            # just the first predicted step
        early_preds.append(y_first.cpu().item())         # scalar

early_preds = np.array(early_preds).reshape(-1, 1)  # shape (16, 1)


In [ ]:
# Rebuild inference dataset and loader for stride=1
infer_set = SeqDataset(test_df, FEAT_COLS, target_col=None,
                       L=LOOKBACK, H=HORIZON, stride=1)

infer_loader = torch.utils.data.DataLoader(
    infer_set, batch_size=256, shuffle=False, drop_last=False
)

later_preds = []

model.eval()
with torch.inference_mode():
    for x, _ in tqdm(infer_loader, desc="Predict (main)"):
        x = x.to(device)
        y_hat = model(x, tgt=None, tf_ratio=0.0)  # shape (B, HORIZON)
        later_preds.append(y_hat[:, 0].cpu().numpy())  # first step from each

later_preds = np.concatenate(later_preds).reshape(-1, 1)  # shape (35008, 1)


In [ ]:
tail_preds = []

with torch.inference_mode():
    for i in range(1, HORIZON):  # from 1 to 15
        # Take last LOOKBACK rows minus i
        seq = test_df[FEAT_COLS].iloc[-(LOOKBACK+i):-i].values.astype(np.float32)
        pad_len = LOOKBACK - seq.shape[0]
        pad = np.zeros((pad_len, len(FEAT_COLS)), dtype=np.float32)
        x_padded = np.vstack([pad, seq])  # shape (16, F)
        x_tensor = torch.from_numpy(x_padded).unsqueeze(0).to(device)

        y_hat = model(x_tensor, tgt=None, tf_ratio=0.0)
        tail_preds.append(y_hat[0, 0].cpu().item())

tail_preds = np.array(tail_preds).reshape(-1, 1)  # shape (15, 1)


In [ ]:
# Total: 16 (early) + 34993 (main) + 15 (tail) = 35024
pred_scaled_full = np.vstack([early_preds, later_preds, tail_preds])
assert pred_scaled_full.shape[0] == len(test_df), \
    f"Still mismatch: {pred_scaled_full.shape[0]} vs {len(test_df)}"


In [ ]:
# Inverse scale
pred_kw = y_scaler.inverse_transform(pred_scaled_full).ravel()

# Build submission
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "predictions": pred_kw.ravel()
})

submission.to_csv(DATA_DIR / "seq2seq_predictions_5.csv", index=False)
print("✅ Saved predictions to:", DATA_DIR / "submission.csv")

In [ ]:
model.eval()
predictions = []

with torch.inference_mode():
    for x, _ in tqdm(infer_loader, desc="Predict"):
        x = x.to(device)
        y_hat = model(x, tgt=None, tf_ratio=0.0)   # y_hat: (B, HORIZON)
        y_first = y_hat[:, 0]                      # take only first step
        predictions.append(y_first.cpu().numpy())

pred_scaled = np.concatenate(predictions).reshape(-1, 1)


In [ ]:
# Pad the beginning
pad = np.full((LOOKBACK, 1), np.nan)  # you could also use 0.0 here if needed
pred_scaled_full = np.vstack([pad, pred_scaled])

assert pred_scaled_full.shape[0] == len(test_df), \
    f"Shape mismatch: {pred_scaled_full.shape[0]} vs {len(test_df)}"


In [ ]:
pred_kw = y_scaler.inverse_transform(pred_scaled_full).ravel()

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "puissance_cvac_future_pred": pred_kw
})

submission.to_csv(DATA_DIR / "submission.csv", index=False)
print("Saved prediction file:", submission.shape)


In [ ]:
# window stride = HORIZON (16) -> exactly 2 189 slices, no overlap
infer_set   = SeqDataset(test_df, FEAT_COLS, target_col=None,
                         L=LOOKBACK, H=HORIZON, stride=HORIZON)
# infer_set   = SeqDataset(test_df, FEAT_COLS, "puissance_cvac_future",
#                          LOOKBACK, HORIZON, stride=HORIZON)
# infer_loader = torch.utils.data.DataLoader(infer_set, batch_size=256,
#                                            shuffle=False, drop_last=False)

infer_loader = torch.utils.data.DataLoader(
    infer_set, batch_size=256, shuffle=False, drop_last=False
)


model.eval()
predictions = []
with torch.inference_mode():
# inference loop
    for x, _ in tqdm(infer_loader):
        x = x.to(device)
        y_hat = model(x, tgt=None, tf_ratio=0.0)
        predictions.append(y_hat[:, 0].cpu().numpy())  # only first step

# pred_scaled = np.vstack(predictions).reshape(-1,1)[:len(test_df)]
# pred_kw     = y_scaler.inverse_transform(pred_scaled).ravel()

# submission  = pd.DataFrame({
#     "ID":   test_df["ID"],
#     "puissance_cvac_future_pred": pred_kw
# })
# submission.to_csv(DATA_DIR/"submission.csv", index=False)
# print("Saved predictions:", submission.shape)


pred_scaled = np.concatenate(predictions, axis=0).reshape(-1)

print("Predictions:", pred_scaled.shape)
print("Expected:", len(test_df))

if pred_scaled.shape[0] != len(test_df):
    print("Padding with NaNs to match test set length.")
    pad_len = len(test_df) - pred_scaled.shape[0]
    pred_scaled = np.concatenate([pred_scaled, np.full(pad_len, np.nan)])


# # Flatten prediction list
# pred_scaled = np.concatenate(predictions, axis=0).reshape(-1)

# Safety check
assert len(pred_scaled) == len(test_df), f"Expected {len(test_df)} predictions, got {len(pred_scaled)}"

# Invert scaling
pred_kw = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

# Build submission
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "puissance_cvac_future_pred": pred_kw
})


In [ ]:
submission.head(20)